# 已选训练路径的风险门对齐诊断

只读完成的 Qlib Recorder `52f40778cbe64ca6a5864da356b960d0`。对本轮**已经选定**的 75 个年度/种子训练路径与 25 个集成训练路径，在同样日期、相同10/25bp费用下应用原有 SOXX MaxDD / CVaR5 门。

目的：区分“训练 utility 更好，但训练时已不满足最终尾部要求”与“训练尾部合格、样本外发生风险变化”。不查看其他 checkpoint、不选择新 epoch、不训练、不生成任何新的模型策略路径。SOXX 基准独立按原会计函数构建，仅用于验证和同窗比较。

所有训练窗从现金开始，包含全部入场与后续费用。MaxDD 含初始资金标记；CVaR5 使用现有精确 5% 尾部质量计算。原模型选择、资格结论和核心路径均不变。

In [1]:
import os,sys,json,hashlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
ROOT=Path('/Users/rziiiii/dev/workspace-worktrees/benchmark-anchor-utility-20261002')
ROOT=ROOT if ROOT.is_dir() else Path('/Users/rziiiii/dev/workspace')
sys.path.insert(0,str(ROOT))
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from IPython.display import display,Markdown
from src.workflow.stable_excess_integration import benchmark_path
from src.analysis.integrated_allocation import paired_metrics,validate_replay
summary=json.loads((ROOT/'research/benchmark_anchor_utility_20261002/summary.json').read_text())
assert summary['recorder_id']=='52f40778cbe64ca6a5864da356b960d0'
for name in ('src/analysis/integrated_allocation.py','src/workflow/stable_excess_integration.py'):
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==summary['source_sha256'][name]
qlib.init(provider_uri=str(ROOT/summary['provider_uri']),region='us',kernels=1)
with R.uri_context(summary['recorder_uri']):
    rec=R.get_recorder(experiment_name=summary['experiment'],recorder_id=summary['recorder_id']);assert rec.status=='FINISHED'
    seeds=rec.load_object('analysis/training_diagnostics.pkl')
    ensembles=rec.load_object('analysis/ensemble_audits.pkl')
    development_folds=rec.load_object('inputs/development_fold_inputs.pkl')
    diagnostic_folds=rec.load_object('inputs/diagnostic_fold_inputs.pkl')
    development_models=rec.load_object('models/development.pkl')
    diagnostic_models=rec.load_object('models/selected_annual_refits.pkl')
folds=development_folds|diagnostic_folds
models={name:dict(items) for name,items in development_models.items()}
for name,items in diagnostic_models.items():models.setdefault(name,{}).update(items)
assert len(seeds)==75 and len(ensembles)==25 and len(folds)==25
print({'source_recorder':rec.id,'selected_seed_paths':len(seeds),'selected_ensemble_paths':len(ensembles),'costs':[10,25],'market_fits':False,'checkpoint_search':False})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[483:MainThread](2026-10-03 00:29:16,775) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[483:MainThread](2026-10-03 00:29:16,776) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[483:MainThread](2026-10-03 00:29:16,776) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-worktrees/benchmark-anchor-utility-20261002/data/qlib/published_futu_affine_20260925_v1')}


{'source_recorder': '52f40778cbe64ca6a5864da356b960d0', 'selected_seed_paths': 75, 'selected_ensemble_paths': 25, 'costs': [10, 25], 'market_fits': False, 'checkpoint_search': False}


## 只重新标注并分析已保存的训练路径

从对应原生 Dataset 年度输入取得 signal/trade/exit 日期，把它们与已保存的 net_equity、fees、turnover 数组绑定。核对训练特征和回报哈希、数组行数及财富乘积；不调用网络或策略回放。

`paired_metrics` 内部调用 Qlib `risk_analysis(..., N=252, mode="product")`，并保留初始资金锚定 MaxDD 和原精确 CVaR5。两个风险门使用原始1e−10数值容差，不修改门槛。

In [2]:
rows_out=[];benchmark_cache={};max_benchmark_diff=0.;max_equity_diff=0.
for kind,entries in [('seed',seeds),('ensemble',ensembles)]:
    for entry in entries:
        name,year=entry['variant'],int(entry['year']);fold=folds[(name,year)]
        dates=fold['rows'][['signal_date','trade_date','exit_date']].reset_index(drop=True)
        fhash=hashlib.sha256(np.asarray(fold['features'],dtype=np.float64).tobytes()).hexdigest()
        rhash=hashlib.sha256(np.asarray(fold['returns'],dtype=np.float64).tobytes()).hexdigest()
        record=models[name][year]
        assert fhash==record['training_feature_sha256'] and rhash==record['training_return_sha256']
        if kind=='seed':
            assert entry['failure'] is None
            assert entry['training_feature_sha256']==fhash and entry['training_return_sha256']==rhash
        for cost in (10,25):
            arr=entry['training_paths'][cost]
            assert len(arr['net_equity'])==len(dates)
            path=dates.copy()
            path['strategy_return']=np.asarray(arr['net_equity'])-1
            path['fees']=arr['fees'];path['turnover']=arr['turnover'];path['cost_bps']=cost
            path['equity']=arr['equity']
            path=validate_replay(path)
            max_equity_diff=max(max_equity_diff,float(np.max(np.abs(path.equity.to_numpy()-np.cumprod(arr['net_equity'])))))
            key=(name,year,cost)
            if key not in benchmark_cache:benchmark_cache[key]=benchmark_path(fold['rows'],cost_bps=cost)
            benchmark=benchmark_cache[key]
            if kind=='seed':
                diff=float(np.max(np.abs(np.asarray(entry['benchmark_net_equity'][cost])-(1+benchmark.strategy_return.to_numpy()))))
                max_benchmark_diff=max(max_benchmark_diff,diff);assert diff<1e-12
            m=paired_metrics(path,benchmark)
            dd_gap=m['max_drawdown_pct']-m['benchmark_max_drawdown_pct']
            cvar_gap=m['cvar5_pct']-m['benchmark_cvar5_pct']
            dd_ok=dd_gap>=-1e-10;cvar_ok=cvar_gap>=-1e-10
            j=entry['centered_worst_utility'];assert j>=0
            rows_out.append({'variant':name,'model_year':year,'evaluation_role':'development_fold' if year<=2022 else 'reused_diagnostic_fold','kind':kind,'seed':str(entry['seed']) if kind=='seed' else 'ensemble','cost_bps':cost,'selected_epoch':str(entry['selected_epoch']) if kind=='seed' else 'fixed_three_seed_ensemble','selected_J':float(j),'centered_utility_at_cost':float(entry['centered_utility_by_cost'][cost]),'training_rows':len(path),'training_feature_sha256':fhash,'training_return_sha256':rhash,'maxdd_gap_pp':dd_gap,'cvar5_gap_pp':cvar_gap,'maxdd_gate_pass':bool(dd_ok),'cvar5_gate_pass':bool(cvar_ok),'both_tail_gates_pass':bool(dd_ok and cvar_ok),**m})
table=pd.DataFrame(rows_out)
assert len(table)==200
assert max_equity_diff<1e-12
counts=table.groupby(['variant','evaluation_role','kind','cost_bps']).agg(paths=('both_tail_gates_pass','size'),maxdd_pass=('maxdd_gate_pass','sum'),cvar_pass=('cvar5_gate_pass','sum'),both_pass=('both_tail_gates_pass','sum'),worst_maxdd_gap_pp=('maxdd_gap_pp','min'),worst_cvar5_gap_pp=('cvar5_gap_pp','min'),minimum_selected_J=('selected_J','min')).reset_index()
display(counts.round(7))
print({'stored_vs_independent_benchmark_multiplier_max_abs':max_benchmark_diff,'stored_equity_identity_max_abs':max_equity_diff,'existing_selected_policy_paths_only':True})

,variant,evaluation_role,kind,cost_bps,paths,maxdd_pass,cvar_pass,both_pass,worst_maxdd_gap_pp,worst_cvar5_gap_pp,minimum_selected_J
0,constant,development_fold,ensemble,10,7,5,5,5,-0.015993,-0.001729,0.000008
1,constant,development_fold,ensemble,25,7,5,5,5,-0.016407,-0.001732,0.000008
2,constant,development_fold,seed,10,21,15,15,15,-0.015993,-0.001729,0.000008
3,constant,development_fold,seed,25,21,15,15,15,-0.016407,-0.001732,0.000008
4,price,development_fold,ensemble,10,7,0,0,0,-10.519937,-0.858393,0.008209
5,price,development_fold,ensemble,25,7,0,0,0,-10.595187,-0.862776,0.008209
6,price,development_fold,seed,10,21,0,0,0,-12.731644,-1.064555,0.007279
7,price,development_fold,seed,25,21,0,0,0,-12.836209,-1.069958,0.007279
8,price,reused_diagnostic_fold,ensemble,10,4,1,0,0,-1.001750,-0.468307,0.005846
9,price,reused_diagnostic_fold,ensemble,25,4,0,0,0,-1.358092,-0.471547,0.005846


{'stored_vs_independent_benchmark_multiplier_max_abs': 2.220446049250313e-16, 'stored_equity_identity_max_abs': 0.0, 'existing_selected_policy_paths_only': True}


## 按年度列出集成模型

这些是每个模型自己曾用于选定 incumbent 的训练历史，不是该年份的样本外结果。相邻年度训练窗大量重叠，不能把路径数当独立统计样本。

In [3]:
ensemble_table=table.loc[table.kind.eq('ensemble'),['variant','model_year','cost_bps','training_rows','selected_J','max_drawdown_pct','benchmark_max_drawdown_pct','maxdd_gap_pp','cvar5_pct','benchmark_cvar5_pct','cvar5_gap_pp','both_tail_gates_pass']]
display(ensemble_table.round(6))
worst=table.sort_values('maxdd_gap_pp').head(6)[['variant','model_year','kind','seed','cost_bps','selected_J','maxdd_gap_pp','cvar5_gap_pp']]
display(worst.round(6))

,variant,model_year,cost_bps,training_rows,selected_J,max_drawdown_pct,benchmark_max_drawdown_pct,maxdd_gap_pp,cvar5_pct,benchmark_cvar5_pct,cvar5_gap_pp,both_tail_gates_pass
150,constant,2016,10,1256,0.000040,-31.595437,-31.595437,-0.000000,-3.263669,-3.263669,0.000000,True
151,constant,2016,25,1256,0.000040,-31.595437,-31.595437,0.000000,-3.263669,-3.263669,0.000000,True
152,price,2016,10,1256,0.008382,-31.824477,-31.595437,-0.229040,-3.559419,-3.263669,-0.295751,False
153,price,2016,25,1256,0.008382,-31.999660,-31.595437,-0.404223,-3.561582,-3.263669,-0.297913,False
154,price_vol,2016,10,1256,0.009635,-31.288215,-31.595437,0.307221,-3.574232,-3.263669,-0.310563,False
155,price_vol,2016,25,1256,0.009635,-31.463598,-31.595437,0.131838,-3.576436,-3.263669,-0.312768,False
156,constant,2017,10,1508,0.000014,-31.609963,-31.595437,-0.014526,-3.269053,-3.267419,-0.001634,False
157,constant,2017,25,1508,0.000014,-31.610256,-31.595437,-0.014819,-3.269055,-3.267419,-0.001636,False
158,price,2017,10,1508,0.013157,-33.195228,-31.595437,-1.599791,-3.758690,-3.267419,-0.491272,False
159,price,2017,25,1508,0.013157,-33.444808,-31.595437,-1.849372,-3.761617,-3.267419,-0.494198,False


,variant,model_year,kind,seed,cost_bps,selected_J,maxdd_gap_pp,cvar5_gap_pp
115,price,2022,seed,17,25,0.016563,-12.836209,-1.069958
121,price_vol,2022,seed,17,25,0.019273,-12.734470,-1.206135
114,price,2022,seed,17,10,0.016563,-12.731644,-1.064555
120,price_vol,2022,seed,17,10,0.019273,-12.628429,-1.200039
125,price_vol,2022,seed,71,25,0.016852,-12.175934,-1.128411
124,price_vol,2022,seed,71,10,0.016852,-12.080223,-1.122796


In [4]:
out=Path('/tmp')
table.to_csv(out/'benchmark_anchor_training_risk.csv',index=False)
payload={'source_recorder_id':summary['recorder_id'],'source_status':'FINISHED','no_market_fits':True,'no_new_model_policy_replay':True,'no_checkpoint_search':True,'selection_or_qualification_changed':False,'selected_seed_models':75,'selected_ensemble_models':25,'costs':[10,25],'total_rows':200,'metric_source':'src.analysis.integrated_allocation.paired_metrics; native Qlib risk_analysis N252 modeproduct; initialcapital anchoredMaxDD and existing exactCVaR5','checks':{'stored_benchmark_vs_independent_max_abs':max_benchmark_diff,'stored_equity_identity_max_abs':max_equity_diff},'counts':counts.to_dict('records'),'metrics':table.to_dict('records'),'limitations':['These are repeated/overlapping training histories, not new independent evidence.','A positive realized-sequence utility does not enforce daily CVaR or path MaxDD constraints.','Training violation establishes objective/gate misalignment on selected paths, not proof that a constrained learner would generalize or earn alpha.','No different epochs, coefficients, gates or candidates were selected.']}
(out/'benchmark_anchor_training_risk.json').write_text(json.dumps(payload,indent=2,default=str,allow_nan=False))
print('Saved the200-row diagnostic CSV and source-backed JSON; original research artifacts unchanged.')

Saved the200-row diagnostic CSV and source-backed JSON; original research artifacts unchanged.


In [5]:
lines=['## 中文结论','', '已审查当前选中的75个种子模型与25个集成模型，共200条成本对应训练路径。全部来自原Recorder保存的完整训练账本，没有寻找其他checkpoint或重新回放模型。','']
for name in ('constant','price','price_vol'):
    group=table.loc[table.variant.eq(name)&table.evaluation_role.eq('development_fold')&table.kind.eq('ensemble')]
    lines.append(f'- {name}：开发期7个年度集成、两档成本共{len(group)}项，训练内MaxDD通过{int(group.maxdd_gate_pass.sum())}项，CVaR5通过{int(group.cvar5_gate_pass.sum())}项，两者同时通过{int(group.both_tail_gates_pass.sum())}项；最差MaxDD/CVaR相对SOXX差距分别为{group.maxdd_gap_pp.min():.3f}/{group.cvar5_gap_pp.min():.3f}个百分点。')
conditioned=table.loc[table.variant.isin(['price','price_vol'])&table.evaluation_role.eq('development_fold')]
violations=int((~conditioned.both_tail_gates_pass).sum())
lines+=['',f'条件信息模型在开发年度的已选训练路径中，有{violations}/{len(conditioned)}项已不满足至少一个最终尾部门，尽管保存的最差成本训练J均不低于基准。']
if violations:
    lines.append('因此，至少存在可直接验证的训练目标与最终风险门不一致；不能把全部失败解释为只在样本外才出现的风险漂移。这不排除样本外变化，也不证明改成受约束训练就能产生稳定超额。')
else:
    lines.append('本次没有发现这些已选训练路径的风险门失配；后续需要解释样本外风险变化，但不得据此选择新阈值。')
lines+=['','原模型选择和历史资格仍不变。本诊断只为下一项独立登记提供证据，不选择不同epoch或改动任何门槛。']
display(Markdown(chr(10).join(lines)))

## 中文结论

已审查当前选中的75个种子模型与25个集成模型，共200条成本对应训练路径。全部来自原Recorder保存的完整训练账本，没有寻找其他checkpoint或重新回放模型。

- constant：开发期7个年度集成、两档成本共14项，训练内MaxDD通过10项，CVaR5通过10项，两者同时通过10项；最差MaxDD/CVaR相对SOXX差距分别为-0.016/-0.002个百分点。
- price：开发期7个年度集成、两档成本共14项，训练内MaxDD通过0项，CVaR5通过0项，两者同时通过0项；最差MaxDD/CVaR相对SOXX差距分别为-10.595/-0.863个百分点。
- price_vol：开发期7个年度集成、两档成本共14项，训练内MaxDD通过2项，CVaR5通过0项，两者同时通过0项；最差MaxDD/CVaR相对SOXX差距分别为-10.906/-1.049个百分点。

条件信息模型在开发年度的已选训练路径中，有112/112项已不满足至少一个最终尾部门，尽管保存的最差成本训练J均不低于基准。
因此，至少存在可直接验证的训练目标与最终风险门不一致；不能把全部失败解释为只在样本外才出现的风险漂移。这不排除样本外变化，也不证明改成受约束训练就能产生稳定超额。

原模型选择和历史资格仍不变。本诊断只为下一项独立登记提供证据，不选择不同epoch或改动任何门槛。